# Practical 7: ETL Pipelines: CSV/JSON Extraction, Validation & Incremental Loading

**Name:** Vedant Vinod Rakshe

**Aim:** To design and implement ETL pipelines that extract data from single and multiple CSV files and JSON documents, clean and transform it, identify and remove invalid records, validate data before loading, load it into a relational database and perform incremental loads instead of reloading the entire dataset.

## Task 1: Setup - Sample Source Files and Target Database

In [1]:
# Name: Vedant Vinod Rakshe
print("Name: Vedant Vinod Rakshe")
import os, glob, json, sqlite3
import pandas as pd

# --- three monthly sales files (contain dirty data on purpose) ---
with open("sales_jan.csv", "w") as f:
    f.write("order_id,customer_id,product,category,quantity,price,order_date\n")
    f.write("1001,C01,  laptop ,electronics,1,55000,2026-01-03\n")
    f.write("1002,C02,Mouse,Electronics,2,500,2026-01-05\n")
    f.write("1003,C03,desk lamp,home,1,,2026-01-07\n")            # missing price
    f.write("1004,C01,Keyboard,electronics,1,1500,2026-01-09\n")
    f.write("1004,C01,Keyboard,electronics,1,1500,2026-01-09\n")  # duplicate row
    f.write("1005,C04,Notebook,stationery,5,60,2026-01-12\n")
with open("sales_feb.csv", "w") as f:
    f.write("order_id,customer_id,product,category,quantity,price,order_date\n")
    f.write("1006,C02,Monitor,Electronics,1,12000,2026-02-02\n")
    f.write("1007,C05,Chair,furniture,2,3500,2026-02-06\n")
    f.write("1008,C03,Pen Pack,Stationery,-3,40,2026-02-10\n")    # negative quantity
    f.write("1009,C06,Headphones,electronics,1,2500,2026-02-14\n")
with open("sales_mar.csv", "w") as f:
    f.write("order_id,customer_id,product,category,quantity,price,order_date\n")
    f.write("1010,C01,Backpack,fashion,1,1800,2026-03-01\n")
    f.write("1011,C07,Table,furniture,1,9000,2026-03-05\n")
    f.write("1012,C04,Mouse,electronics,3,500,not-a-date\n")       # invalid date
    f.write("1013,C02,Notebook,stationery,10,60,2026-03-20\n")
    f.write("1004,C01,Keyboard,electronics,1,1500,2026-01-09\n")  # repeat of an earlier order

# --- JSON orders with nested customer details ---
def order(oid, cid, name, city, sales, qty, status):
    return {"order_id": oid, "customer": {"id": cid, "name": name, "city": city},
            "sales": sales, "quantity": qty, "status": status}

orders = [
    order(2001, "C01", "Amit Shah", "Mumbai", 55000.0, 1, "delivered"),
    order(2002, "C02", "Riya Patil", "Pune", 1000.0, 2, "delivered"),
    order(2003, "C03", "Neha Rao", "Delhi", 3000.0, 2, "shipped"),
    order(2004, None, "Unknown", None, 700.0, 1, "placed"),          # missing customer
    order(2005, "C04", "Karan Jain", "Nashik", -250.0, 1, "returned"),  # negative sales
    order(2006, "C01", "Amit Shah", "Mumbai", 1500.0, 0, "shipped"),    # zero quantity
    order(2007, "C05", "Sana Khan", "Pune", 4200.0, 3, "delivered"),
]
with open("orders.json", "w") as f:
    json.dump(orders, f, indent=2)

if os.path.exists("etl_database.db"):
    os.remove("etl_database.db")          # start with a fresh target database
conn = sqlite3.connect("etl_database.db")
print("Created: sales_jan.csv, sales_feb.csv, sales_mar.csv, orders.json, etl_database.db")

Name: Vedant Vinod Rakshe
Created: sales_jan.csv, sales_feb.csv, sales_mar.csv, orders.json, etl_database.db


## Task 2: ETL Pipeline for a Single CSV File (Extract, Clean, Transform, Load)

In [2]:
print("Name: Vedant Vinod Rakshe")
def clean_sales(df):
    """Reusable CLEAN + TRANSFORM step for sales data."""
    df = df.copy()
    df["product"] = df["product"].str.strip().str.title()      # standardise text
    df["category"] = df["category"].str.strip().str.title()
    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
    df = df.drop_duplicates(subset=[c for c in df.columns if c != "source_file"])
    df["total_amount"] = df["quantity"] * df["price"]            # derived column
    return df

# EXTRACT
raw = pd.read_csv("sales_jan.csv")
print("Extracted rows :", len(raw))

# TRANSFORM
jan = clean_sales(raw)
jan["order_month"] = jan["order_date"].dt.strftime("%Y-%m")
print("After cleaning :", len(jan), "rows (duplicate removed)\n")

# LOAD
jan.to_sql("sales_jan", conn, if_exists="replace", index=False)
print(pd.read_sql("SELECT order_id, product, category, total_amount FROM sales_jan", conn))

Name: Vedant Vinod Rakshe
Extracted rows : 6
After cleaning : 5 rows (duplicate removed)

   order_id    product     category  total_amount
0      1001     Laptop  Electronics       55000.0
1      1002      Mouse  Electronics        1000.0
2      1003  Desk Lamp         Home           NaN
3      1004   Keyboard  Electronics        1500.0
4      1005   Notebook   Stationery         300.0


## Task 3: Extract Data from Multiple CSV Files and Combine into One Dataset

In [3]:
print("Name: Vedant Vinod Rakshe")
frames = []
for path in sorted(glob.glob("sales_*.csv")):
    part = pd.read_csv(path)
    part["source_file"] = os.path.basename(path)       # keep lineage information
    print(f"{path:<14} -> {len(part)} rows")
    frames.append(part)

combined = pd.concat(frames, ignore_index=True)
print("Rows after combining   :", len(combined))
combined = clean_sales(combined)
print("Rows after de-duplicate:", len(combined))

combined.to_sql("sales_all", conn, if_exists="replace", index=False)
print()
print(pd.read_sql("""SELECT source_file, COUNT(*) AS rows_loaded
                    FROM sales_all GROUP BY source_file""", conn))

Name: Vedant Vinod Rakshe
sales_feb.csv  -> 4 rows
sales_jan.csv  -> 6 rows
sales_mar.csv  -> 5 rows
Rows after combining   : 15
Rows after de-duplicate: 13

     source_file  rows_loaded
0  sales_feb.csv            4
1  sales_jan.csv            5
2  sales_mar.csv            4


## Task 4: Extract JSON Data, Transform Selected Fields and Load to the Database

In [4]:
print("Name: Vedant Vinod Rakshe")
def transform_orders(records):
    """Flatten nested JSON and keep only the selected fields."""
    df = pd.json_normalize(records).rename(columns={
        "customer.id": "customer_id", "customer.name": "customer_name", "customer.city": "city"})
    df = df[["order_id", "customer_id", "customer_name", "city", "sales", "quantity", "status"]]
    df["status"] = df["status"].str.upper()
    df["sales"] = df["sales"].round(2)
    return df

with open("orders.json") as f:
    raw_json = json.load(f)                            # EXTRACT
stg = transform_orders(raw_json)                       # TRANSFORM
stg.to_sql("stg_orders", conn, if_exists="replace", index=False)   # LOAD into staging table
print(pd.read_sql("SELECT * FROM stg_orders", conn).to_string(index=False))

Name: Vedant Vinod Rakshe
 order_id customer_id customer_name   city   sales  quantity    status
     2001         C01     Amit Shah Mumbai 55000.0         1 DELIVERED
     2002         C02    Riya Patil   Pune  1000.0         2 DELIVERED
     2003         C03      Neha Rao  Delhi  3000.0         2   SHIPPED
     2004         NaN       Unknown    NaN   700.0         1    PLACED
     2005         C04    Karan Jain Nashik  -250.0         1  RETURNED
     2006         C01     Amit Shah Mumbai  1500.0         0   SHIPPED
     2007         C05     Sana Khan   Pune  4200.0         3 DELIVERED


## Task 5: Identify and Remove Invalid Records

Rules: customer_id must not be null, sales must be greater than 0 and quantity must be greater than 0. Rejected records are saved with the reason in a separate table.

In [5]:
print("Name: Vedant Vinod Rakshe")
def find_invalid(row):
    problems = []
    if pd.isna(row["customer_id"]): problems.append("missing customer_id")
    if row["sales"] <= 0:           problems.append("non-positive sales")
    if row["quantity"] <= 0:        problems.append("non-positive quantity")
    return "; ".join(problems)

staged = pd.read_sql("SELECT * FROM stg_orders", conn)
staged["reason"] = staged.apply(find_invalid, axis=1)

df_invalid = staged[staged["reason"] != ""]
df_valid = staged[staged["reason"] == ""].drop(columns="reason")

print(f"Valid records   : {len(df_valid)}")
print(f"Invalid records : {len(df_invalid)} (removed)\n")
print(df_invalid[["order_id", "reason"]].to_string(index=False))

df_invalid.to_sql("rejected_orders", conn, if_exists="replace", index=False)   # audit trail

Name: Vedant Vinod Rakshe
Valid records   : 4
Invalid records : 3 (removed)

 order_id                reason
     2004   missing customer_id
     2005    non-positive sales
     2006 non-positive quantity


## Task 6: Data Validation Before Loading into the Target Database

The target tables use primary keys, a foreign key and CHECK constraints. A validation gate runs before every load and aborts the load if the batch has structural problems.

In [6]:
print("Name: Vedant Vinod Rakshe")
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript("""
DROP TABLE IF EXISTS orders;  DROP TABLE IF EXISTS customers;
CREATE TABLE customers (customer_id TEXT PRIMARY KEY, customer_name TEXT, city TEXT);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id TEXT NOT NULL REFERENCES customers(customer_id),
    sales REAL CHECK (sales > 0), quantity INTEGER CHECK (quantity > 0), status TEXT);
CREATE TABLE IF NOT EXISTS etl_audit (
    run_no INTEGER PRIMARY KEY AUTOINCREMENT, rows_loaded INTEGER, rows_skipped INTEGER);
""")
REQUIRED = ["order_id", "customer_id", "customer_name", "city", "sales", "quantity", "status"]

def validate_before_load(df):
    """Returns a list of problems; an empty list means the batch may be loaded."""
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        return [f"missing columns {missing}"]
    errors = []
    if df["order_id"].duplicated().any():          errors.append("duplicate order_id values")
    if df["customer_id"].isna().any():             errors.append("null customer_id")
    if (df["sales"] <= 0).any() or (df["quantity"] <= 0).any():
        errors.append("non-positive sales or quantity")
    return errors

def load_orders(df):
    errors = validate_before_load(df)
    if errors:
        print("  VALIDATION FAILED -> load aborted:", errors)
        return 0
    print("  Validation passed -> loading", len(df), "records")
    cust = df[["customer_id", "customer_name", "city"]].drop_duplicates("customer_id")
    known = set(pd.read_sql("SELECT customer_id FROM customers", conn)["customer_id"])
    cust[~cust["customer_id"].isin(known)].to_sql("customers", conn, if_exists="append", index=False)
    df[["order_id", "customer_id", "sales", "quantity", "status"]].to_sql(
        "orders", conn, if_exists="append", index=False)
    return len(df)

print("Batch 1 (a required column is missing):")
load_orders(df_valid.drop(columns=["sales"]))
print("Batch 2 (duplicate order id):")
load_orders(pd.concat([df_valid, df_valid.head(1)]))
print("Batch 3 (clean data):")
load_orders(df_valid)

print("\nTarget table: customers")
print(pd.read_sql("SELECT * FROM customers", conn).to_string(index=False))
print("\nTarget table: orders")
print(pd.read_sql("SELECT * FROM orders", conn).to_string(index=False))

Name: Vedant Vinod Rakshe
Batch 1 (a required column is missing):
  VALIDATION FAILED -> load aborted: ["missing columns ['sales']"]
Batch 2 (duplicate order id):
  VALIDATION FAILED -> load aborted: ['duplicate order_id values']
Batch 3 (clean data):
  Validation passed -> loading 4 records

Target table: customers
customer_id customer_name   city
        C01     Amit Shah Mumbai
        C02    Riya Patil   Pune
        C03      Neha Rao  Delhi
        C05     Sana Khan   Pune

Target table: orders
 order_id customer_id   sales  quantity    status
     2001         C01 55000.0         1 DELIVERED
     2002         C02  1000.0         2 DELIVERED
     2003         C03  3000.0         2   SHIPPED
     2007         C05  4200.0         3 DELIVERED


## Task 7: Incremental Data Loading

Instead of reloading everything, only orders whose order_id is not yet in the target table are loaded. Every run is logged in an audit table.

In [7]:
# A second file arrives: 2 old orders (already loaded), 2 new orders, 1 invalid order
print("Name: Vedant Vinod Rakshe")
new_orders = [
    order(2002, "C02", "Riya Patil", "Pune", 1000.0, 2, "delivered"),
    order(2007, "C05", "Sana Khan", "Pune", 4200.0, 3, "delivered"),
    order(2008, "C06", "Dev Mehta", "Surat", 2600.0, 2, "placed"),
    order(2009, "C01", "Amit Shah", "Mumbai", 899.0, 1, "shipped"),
    order(2010, "C07", "Tina Dsouza", "Goa", -5.0, 1, "placed"),
]
with open("orders_new.json", "w") as f:
    json.dump(new_orders, f)

def incremental_load(path):
    new = transform_orders(json.load(open(path)))
    loaded_ids = set(pd.read_sql("SELECT order_id FROM orders", conn)["order_id"])
    fresh = new[~new["order_id"].isin(loaded_ids)]               # only unseen orders
    skipped = len(new) - len(fresh)
    fresh = fresh[fresh.apply(find_invalid, axis=1) == ""]       # drop invalid ones
    n = load_orders(fresh) if len(fresh) else 0
    conn.execute("INSERT INTO etl_audit (rows_loaded, rows_skipped) VALUES (?, ?)", (n, skipped))
    conn.commit()
    print(f"  Result: {n} new rows loaded, {skipped} existing rows skipped\n")

print("Run 1 (new file arrives):")
incremental_load("orders_new.json")
print("Run 2 (same file again - nothing new):")
incremental_load("orders_new.json")

print("ETL audit log:")
print(pd.read_sql("SELECT * FROM etl_audit", conn).to_string(index=False))
total = conn.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
print("\nTotal orders in target table:", total)

Name: Vedant Vinod Rakshe
Run 1 (new file arrives):
  Validation passed -> loading 2 records
  Result: 2 new rows loaded, 2 existing rows skipped

Run 2 (same file again - nothing new):
  Result: 0 new rows loaded, 4 existing rows skipped

ETL audit log:
 run_no  rows_loaded  rows_skipped
      1            2             2
      2            0             4

Total orders in target table: 6


**Conclusion:** Complete ETL pipelines were built for CSV and JSON sources. Multiple CSV files were combined into one dataset, invalid records were identified and moved to a rejected table, a validation gate stopped bad batches from reaching the database, and incremental loading ensured that only new records were added on each run.